# Pilote MoulSot avec Mistral

Ce notebook transcrit au maximum 200 fichiers audio locaux, reprend automatiquement apres interruption et conserve les sorties brutes. Ces sorties sont des candidats, pas une verite terrain.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT = Path.cwd()
if not (PROJECT / 'transcriber.py').exists():
    PROJECT = Path('C:/Users/sail_/Documents/Codex/2026-08-15/https-github-com-badr-joulali-mistral/mistral_darija_stt_benchmark')
sys.path.insert(0, str(PROJECT))
MOULSOT_ROOT = Path('/home/skiredj.abderrahman/badr/badr_data/omniasr_moulsot80')
OUTPUT_DIR = PROJECT / 'outputs' / 'moulsot_mistral_200'
print(PROJECT)
print(MOULSOT_ROOT)

## Cle API

Colle la cle uniquement dans cette session Jupyter. Ne sauvegarde pas cette cellule avec une vraie cle et ne la pousse jamais vers Git.

In [ ]:
os.environ['MISTRAL_API_KEY'] = 'COLLER_LA_CLE_ICI'
os.environ['MISTRAL_MODEL'] = 'voxtral-mini-latest'
assert os.environ['MISTRAL_API_KEY'] != 'COLLER_LA_CLE_ICI', 'Colle la cle dans cette cellule puis execute-la.'
print('Modele:', os.environ['MISTRAL_MODEL'])
print('Cle configuree:', bool(os.environ['MISTRAL_API_KEY']))

In [ ]:
parquets = sorted(MOULSOT_ROOT.glob('version=*/corpus=*/split=train/language=*/*.parquet'))
print('Fichiers Parquet:', len(parquets))
assert parquets, f'Dataset MoulSot absent: {MOULSOT_ROOT}'

In [ ]:
import subprocess
subprocess.run([sys.executable, str(PROJECT / 'moulsot_mistral_pilot.py'), '--parquet-root', str(MOULSOT_ROOT), '--staging-dir', str(PROJECT / 'outputs/moulsot_mistral_200_audio'), '--output-root', str(OUTPUT_DIR), '--limit', '200'], check=True)

In [ ]:
import json
summary = json.loads((OUTPUT_DIR / 'summary.json').read_text(encoding='utf-8'))
print('Succes:', sum(x.get('status') == 'success' for x in summary))
print('Erreurs:', sum(x.get('status') != 'success' for x in summary))
for row in summary[:10]:
    print(row['sample_id'], ':', row.get('transcription', '')[:180])